In [39]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import random
import time
from datetime import datetime, timedelta

In [40]:
def scrape_airline_data():
    print("Scraping US airline data from Wikipedia...")
    url = "https://en.wikipedia.org/wiki/List_of_largest_airlines_in_North_America"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    tables = soup.find_all("table", class_="wikitable")
    table_dataframes = {}

    for table_index, table in enumerate(tables):
        print(f"\nProcessing table {table_index + 1}...")

        # Read header row dynamically
        header_row = table.find("tr")
        if not header_row:
            print(f"No header found in table {table_index + 1}, skipping...")
            continue

        headers_list = [th.get_text(strip=True) for th in header_row.find_all(["th", "td"])]
        print(f"Columns found: {headers_list}")

        if not headers_list:
            continue

        # Read each row and map to its own header
        rows = table.find_all("tr")[1:]
        table_data = []

        for row in rows:
            cols = row.find_all(["td", "th"])
            if not cols:
                continue

            row_data = {}
            for i, header in enumerate(headers_list):
                row_data[header] = cols[i].get_text(strip=True) if i < len(cols) else "N/A"

            if any(v != "N/A" for v in row_data.values()):
                table_data.append(row_data)

            if table_data:
                df = pd.DataFrame(table_data)
                df.rename(columns={"Airlines": "Airline", "Airline(s)": "Airline"}, inplace=True)

            if table_index == 0:
                df.rename(columns={c: f"ASM_{c}" for c in df.columns if c.isdigit()}, inplace=True)

            elif table_index == 1:
                df.rename(columns={c: f"Passengers_{c}" for c in df.columns if c.isdigit()}, inplace=True)
            
            table_name = f"table_{table_index + 1}"
            table_dataframes[table_name] = df
            # Save each table as its own CSV
            filename = f"airlines_{table_name}.csv"
            df.to_csv(filename, index=False)
        print(f"Saved {len(table_data)} records to {filename}")

    # Merge all tables on Airline column if it exists in all
    common_key = "Airline"
    tables_with_airline = {k: v for k, v in table_dataframes.items() if common_key in v.columns}

    if len(tables_with_airline) > 1:
        print("\nMerging tables on Airline column...")
        merged_df = None
        for name, df in tables_with_airline.items():
            if merged_df is None:
                merged_df = df
            else:
                # Only keep new columns to avoid nulls from mismatched structures
                new_cols = [c for c in df.columns if c not in merged_df.columns]
                merged_df = merged_df.merge(df[["Airline"] + new_cols], on="Airline", how="left")

        merged_df.to_csv("us_airlines.csv", index=False)
        print(f"Saved merged data: {merged_df.shape[0]} rows, {merged_df.shape[1]} columns")
        print(f"Final columns: {list(merged_df.columns)}")
        return merged_df.to_dict("records")

    elif len(tables_with_airline) == 1:
        name, df = list(tables_with_airline.items())[0]
        df.to_csv("us_airlines.csv", index=False)
        return df.to_dict("records")

    else:
        print("No tables with Airline column found.")
        return []

In [41]:
def main():
    # Part 1 — Real scrape with dynamic column mapping
    airlines = scrape_airline_data()

    if airlines:
        airline_df = pd.DataFrame(airlines)
        airline_df.to_csv("us_airlines.csv", index=False)
        print(f"\nSaved {len(airlines)} airline records to us_airlines.csv")
        print(f"Columns captured: {list(airline_df.columns)}")
    else:
        print("No airline data scraped.")

if __name__ == "__main__":
    main()

Scraping US airline data from Wikipedia...

Processing table 1...
Columns found: ['Rank', 'Airline', 'Country', '2025', '2024', '2023', '2022', '2021', '2020', '2019', 'Ref']
Saved 10 records to airlines_table_1.csv

Processing table 2...
Columns found: ['Airlines', 'Country', '2025', '2024', '2023', '2022', '2021', '2020', '2019', 'Ref']
Saved 17 records to airlines_table_2.csv

Processing table 3...
Columns found: ['Airline', 'Destinations']
Saved 24 records to airlines_table_3.csv

Processing table 4...
Columns found: ['Airline', 'Daily departures', 'Ref']
Saved 17 records to airlines_table_4.csv

Merging tables on Airline column...
Saved merged data: 10 rows, 20 columns
Final columns: ['Rank', 'Airline', 'Country', 'ASM_2025', 'ASM_2024', 'ASM_2023', 'ASM_2022', 'ASM_2021', 'ASM_2020', 'ASM_2019', 'Ref', 'Passengers_2025', 'Passengers_2024', 'Passengers_2023', 'Passengers_2022', 'Passengers_2021', 'Passengers_2020', 'Passengers_2019', 'Destinations', 'Daily departures']

Saved 10 a